## TabNet + Optuna + RepeatedStratifiedKFold + Metrics

In [1]:
# =====================================================
# ✅ TabNet + Optuna + RepeatedStratifiedKFold + Metrics
# =====================================================
import pandas as pd
import numpy as np
import torch
import random
import optuna
import torch.optim as optim

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, recall_score,
    roc_auc_score, average_precision_score, cohen_kappa_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from pytorch_tabnet.tab_model import TabNetClassifier

# =====================================================
# 🔒 Set seeds for reproducibility
# =====================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =====================================================
# 1️⃣ Load Data
# =====================================================
df = pd.read_csv(r"C:\Users\dbastola2022\OneDrive - Florida Atlantic University\Academics\Research\Malnutrition\MICS\malnutrition\Dataset\ch.csv")

X = df.drop(columns=['status'])
y = df['status']

# ✅ Create final hold-out set for unbiased testing after tuning
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# =====================================================
# 2️⃣ Transformer: Pandas → NumPy for TabNet
# =====================================================
class ToNumpyTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.to_numpy(dtype=np.float32) if hasattr(X, 'to_numpy') else X.astype(np.float32)

# =====================================================
# 3️⃣ Pipeline: SMOTE → NumPy → TabNet
# =====================================================
pipeline = Pipeline(steps=[
    ('smote', SMOTE(random_state=SEED)),
    ('to_numpy', ToNumpyTransformer()),
    ('clf', TabNetClassifier(verbose=0, seed=SEED,
                             device_name='cuda' if torch.cuda.is_available() else 'cpu'))
])

# =====================================================
# 4️⃣ Define Cross-Validation
# =====================================================
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

# =====================================================
# 5️⃣ Optuna Objective Function
# =====================================================
def objective(trial):
    # 🔥 TabNet hyperparameters
    params = {
        "n_d": trial.suggest_int("n_d", 8, 64),
        "n_a": trial.suggest_int("n_a", 8, 64),
        "n_steps": trial.suggest_int("n_steps", 3, 10),
        "gamma": trial.suggest_float("gamma", 1.0, 2.0),
        "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
        "optimizer_fn": optim.Adam,
        "optimizer_params": {"lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True)},
        "mask_type": trial.suggest_categorical("mask_type", ["entmax", "sparsemax"]),
        "n_shared": trial.suggest_int("n_shared", 1, 3),
        "n_independent": trial.suggest_int("n_independent", 1, 3),
    }

    # ✅ Fit-only hyperparams
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
    virtual_batch_size = trial.suggest_categorical("virtual_batch_size", [64, 128])

    # ✅ Apply params
    pipeline.set_params(**{f"clf__{k}": v for k, v in params.items()})

    auc_scores = []

    for train_idx, val_idx in cv.split(X_train_full, y_train_full):
        X_train_cv, X_val_cv = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
        y_train_cv, y_val_cv = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]

        pipeline.fit(
            X_train_cv, y_train_cv,
            clf__eval_set=[(X_val_cv.to_numpy(dtype=np.float32), y_val_cv.to_numpy())],
            clf__eval_metric=['balanced_accuracy'],
            clf__max_epochs=100,
            clf__patience=10,
            clf__batch_size=batch_size,
            clf__virtual_batch_size=virtual_batch_size,
            clf__drop_last=False,
            clf__num_workers=0
        )

        preds_proba = pipeline.named_steps['clf'].predict_proba(X_val_cv.to_numpy(dtype=np.float32))[:, 1]
        auc_scores.append(roc_auc_score(y_val_cv, preds_proba))

    return np.mean(auc_scores)

# =====================================================
# 6️⃣ Run Optuna Study
# =====================================================
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=20)  # increase to 50+ for more robust tuning

print("\n✅ Best Trial:", study.best_trial)

# =====================================================
# 7️⃣ Extract Best Params for Final Model
# =====================================================
best_params = study.best_trial.params
best_batch_size = best_params.pop("batch_size")
best_virtual_batch_size = best_params.pop("virtual_batch_size")

best_params["optimizer_params"] = {"lr": best_params.pop("lr")}
best_params["seed"] = SEED
best_params["verbose"] = 1

# =====================================================
# 8️⃣ Train Final Model on Full Training Data
# =====================================================
final_pipeline = Pipeline(steps=[
    ('smote', SMOTE(random_state=SEED)),
    ('to_numpy', ToNumpyTransformer()),
    ('clf', TabNetClassifier(**best_params,
                             device_name='cuda' if torch.cuda.is_available() else 'cpu'))
])

final_pipeline.fit(
    X_train_full, y_train_full,
    clf__eval_set=[(X_test.to_numpy(dtype=np.float32), y_test.to_numpy())],
    clf__eval_metric=['balanced_accuracy'],
    clf__max_epochs=200,
    clf__patience=10,
    clf__batch_size=best_batch_size,
    clf__virtual_batch_size=best_virtual_batch_size,
    clf__drop_last=False,
    clf__num_workers=0
)

[I 2025-07-27 12:54:18,971] A new study created in memory with name: no-name-b5fb94cf-a694-4405-8b51-4048551ce1b0



Early stopping occurred at epoch 23 with best_epoch = 13 and best_val_0_balanced_accuracy = 0.5876


c:\Users\dbastola2022\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 25 with best_epoch = 15 and best_val_0_balanced_accuracy = 0.59056


c:\Users\dbastola2022\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


...


Early stopping occurred at epoch 41 with best_epoch = 31 and best_val_0_balanced_accuracy = 0.72694

✅ Best Trial: FrozenTrial(number=16, state=TrialState.COMPLETE, values=[0.8045219507335477], datetime_start=datetime.datetime(2025, 7, 27, 15, 5, 43, 260586), datetime_complete=datetime.datetime(2025, 7, 27, 15, 9, 49, 911880), params={'n_d': 38, 'n_a': 56, 'n_steps': 4, 'gamma': 1.2228512460903929, 'lambda_sparse': 1.3970734935405222e-06, 'lr': 0.005351655001790046, 'mask_type': 'entmax', 'n_shared': 1, 'n_independent': 1, 'batch_size': 512, 'virtual_batch_size': 64}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_d': IntDistribution(high=64, log=False, low=8, step=1), 'n_a': IntDistribution(high=64, log=False, low=8, step=1), 'n_steps': IntDistribution(high=10, log=False, low=3, step=1), 'gamma': FloatDistribution(high=2.0, log=False, low=1.0, step=None), 'lambda_sparse': FloatDistribution(high=0.01, log=True, low=1e-06, step=None), 'lr': FloatDistribution(

c:\Users\dbastola2022\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.94524 | val_0_balanced_accuracy: 0.59283 |  0:00:00s
epoch 1  | loss: 0.70296 | val_0_balanced_accuracy: 0.60509 |  0:00:00s
epoch 2  | loss: 0.66674 | val_0_balanced_accuracy: 0.59515 |  0:00:01s
epoch 3  | loss: 0.64237 | val_0_balanced_accuracy: 0.6353  |  0:00:01s
epoch 4  | loss: 0.61485 | val_0_balanced_accuracy: 0.58984 |  0:00:02s
epoch 5  | loss: 0.61335 | val_0_balanced_accuracy: 0.59069 |  0:00:02s
epoch 6  | loss: 0.59102 | val_0_balanced_accuracy: 0.62343 |  0:00:03s
epoch 7  | loss: 0.57477 | val_0_balanced_accuracy: 0.6271  |  0:00:03s
epoch 8  | loss: 0.56438 | val_0_balanced_accuracy: 0.60837 |  0:00:04s
epoch 9  | loss: 0.54919 | val_0_balanced_accuracy: 0.64693 |  0:00:04s
epoch 10 | loss: 0.54427 | val_0_balanced_accuracy: 0.68325 |  0:00:05s
epoch 11 | loss: 0.54188 | val_0_balanced_accuracy: 0.68174 |  0:00:05s
epoch 12 | loss: 0.53951 | val_0_balanced_accuracy: 0.70702 |  0:00:06s
epoch 13 | loss: 0.53044 | val_0_balanced_accuracy: 0.69157 |  0

c:\Users\dbastola2022\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('to_numpy', ToNumpyTransformer()),
                ('clf',
                 TabNetClassifier(n_d=38,
                                  n_a=56,
                                  n_steps=4,
                                  gamma=1.2228512460903929,
                                  cat_idxs=[],
                                  cat_dims=[],
                                  cat_emb_dim=[],
                                  n_independent=1,
                                  n_shared=1,
                                  epsilon=1e-15,
                                  momentum=0.02,
                                  lambda_sparse=1.3970734935405222e-06,
                                  seed=42,
                                  clip_value=1,
                                  verbose=1,
                                  optimizer_fn=<class 'torch.optim.adam.Adam'>,
                                  optimizer_params={'lr': 0.005351655001790046},
                                  scheduler_fn=None,
                                  scheduler_params={},
                                  mask_type='entmax',
                                  input_dim=15,
                                  output_dim=2,
                                  device_name='cpu',
                                  n_shared_decoder=1,
                                  n_indep_decoder=1,
                                  grouped_features=[]))])

In [2]:
# =====================================================
# 9️⃣ Evaluate on Test Set
# =====================================================
y_pred = final_pipeline.predict(X_test)
y_proba = final_pipeline.predict_proba(X_test)[:, 1]

print("\n🎯 FINAL MODEL PERFORMANCE ON TEST SET 🎯")
print("✅ Best Params:", study.best_params)
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("------------------------------------------------------")
print(classification_report(y_test, y_pred, digits=3))
print("------------------------------------------------------")
print(f"🔥 Test ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"🔥 Test Average Precision: {average_precision_score(y_test, y_proba):.3f}")
print(f"🔥 Test Cohen's Kappa: {cohen_kappa_score(y_test, y_pred):.3f}")
print(f"🔥 Test Recall: {recall_score(y_test, y_pred):.3f}")



🎯 FINAL MODEL PERFORMANCE ON TEST SET 🎯
✅ Best Params: {'n_d': 38, 'n_a': 56, 'n_steps': 4, 'gamma': 1.2228512460903929, 'lambda_sparse': 1.3970734935405222e-06, 'lr': 0.005351655001790046, 'mask_type': 'entmax', 'n_shared': 1, 'n_independent': 1, 'batch_size': 512, 'virtual_batch_size': 64}

📊 Confusion Matrix:
 [[533 174]
 [165 439]]
------------------------------------------------------
              precision    recall  f1-score   support

           0      0.764     0.754     0.759       707
           1      0.716     0.727     0.721       604

    accuracy                          0.741      1311
   macro avg      0.740     0.740     0.740      1311
weighted avg      0.742     0.741     0.742      1311

------------------------------------------------------
🔥 Test ROC-AUC: 0.812
🔥 Test Average Precision: 0.814
🔥 Test Cohen's Kappa: 0.480
🔥 Test Recall: 0.727
